In [1]:
import os
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load data
directory = "data/Indian-monuments/images/train"
base_path = os.path.abspath(directory)
class_folders = os.listdir(base_path)

images = []
labels = []
class_counts = {}

for class_name in class_folders:
    class_path = os.path.join(base_path, class_name)
    for image_file in os.listdir(class_path):
        try:
            image_path = os.path.join(class_path, image_file)
            # Check if the file is a valid image file
            if not os.path.isfile(image_path) or not image_path.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue  # Skip non-image files
            image = tf.keras.preprocessing.image.load_img(image_path, target_size=(299, 299))  # InceptionV3 input size
            image = tf.keras.preprocessing.image.img_to_array(image)
            images.append(image)
            labels.append(class_name)
            class_counts[class_name] = class_counts.get(class_name, 0) + 1
        except Exception as e:
            print(f"Error loading {image_path}: {str(e)}")

X = np.array(images)
y = labels

# Preprocess labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split the data into training, validation, and testing sets
X_train, X_val_test, y_train_encoded, y_val_test_encoded = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y)
X_test, X_val, y_test_encoded, y_val_encoded = train_test_split(X_val_test, y_val_test_encoded, test_size=0.5, random_state=42)

# Data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Define the InceptionV3 model
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(299, 299, 3))

# Freeze the base model
base_model.trainable = False

# Create new model on top
inputs = tf.keras.Input(shape=(299, 299, 3))
x = tf.keras.applications.inception_v3.preprocess_input(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
outputs = tf.keras.layers.Dense(len(label_encoder.classes_), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(train_datagen.flow(X_train, y_train_encoded, batch_size=32),
                    steps_per_epoch=len(X_train) / 32,
                    epochs=15,
                    validation_data=(X_val, y_val_encoded))

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test_encoded)
print(f'Test accuracy: {test_accuracy}')

# Save the model
model.save("inceptionv3_model.h5")

# Predictions
image_path = "C:/Users/Charvi Upreti/Documents/GitHub/Unveiling-India-s-Heritage/data/Indian-monuments/images/train/tajmahal/10.jpg"
img = tf.keras.preprocessing.image.load_img(image_path, target_size=(299, 299))
img_array = tf.keras.preprocessing.image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0) / 255.0
predictions = model.predict(img_array)
predicted_class_index = np.argmax(predictions)
predicted_class = label_encoder.classes_[predicted_class_index]
print(f"Predicted class: {predicted_class}")


87910968/87910968 [==============================] - 35s 0us/step
Epoch 1/15
91/91 [==============================] - 161s 2s/step - loss: 3.0716 - accuracy: 0.0975 - val_loss: 6.0126 - val_accuracy: 0.0981
Epoch 2/15
91/91 [==============================] - 157s 2s/step - loss: 2.9708 - accuracy: 0.1190 - val_loss: 7.1932 - val_accuracy: 0.0981
Epoch 3/15
91/91 [==============================] - 157s 2s/step - loss: 2.9112 - accuracy: 0.1487 - val_loss: 8.1631 - val_accuracy: 0.0981
Epoch 4/15
22/91 [======>.......................] - ETA: 1:47 - loss: 2.9133 - accuracy: 0.1402

KeyboardInterrupt: 